# PRISM / IQM Emerald — verified live demo

**Safety:** No cell sends a hardware job until you explicitly set `ALLOW_HARDWARE_RUN=True`. Do not paste tokens into cells or share them in chat. The organizer's own notebook may already define `backend`; run its official setup first.

The 49-gate circuit is phase-only, so we surround **both** circuits with H gates on one qubit to expose phase in an output histogram. Counts are noisy experimental evidence, **not** a proof of whole-unitary equivalence.


## 1. Install required software only if missing

In a separate notebook cell, if needed: `%pip install 'qiskit[visualization]' 'iqm-client[qiskit]'`. Do not force-reinstall packages in a hackathon-managed environment when already available.

In [ ]:
import json
from pathlib import Path
from submission import optimize, _unitary, _equal_unitaries, cost, metrics
from run_experiments import cases
from iqm_emerald_demo import compare_on_backend, ideal_probabilities
cfg = {'w1': 1.0, 'w2': 6.0, 'wd': 0.2}
gauntlet = next(x['circuit'] for x in cases() if x['name'] == 'L3-parity-gauntlet-49-to-1')
optimized = optimize(gauntlet, cfg)
assert _equal_unitaries(_unitary(gauntlet['gates'], 4), _unitary(optimized['gates'], 4))
print('Input gates:', len(gauntlet['gates']))
print('Output:', optimized['gates'])
print('Costs:', cost(gauntlet['gates'], 4, cfg), '->', cost(optimized['gates'], 4, cfg))
print('H-T-H probe ideal probabilities:', ideal_probabilities(gauntlet, probe_qubit=1))


## 2. Verify that an authorized backend already exists

**Option A:** Execute the hackathon's existing IBM/IQM initialization cells; they should define `backend`. **Option B (only with separately authorized direct IQM access):** set `IQM_SERVER_URL` and `IQM_TOKEN` in your environment's secret store, then call `from iqm_emerald_demo import connect_iqm_from_environment; backend = connect_iqm_from_environment()`. Do not display or save tokens. If an event uses an IBM Runtime Sampler rather than a backend with `.run`, consult its provided adapter.


In [ ]:
if 'backend' not in globals():
    print('No backend found. Run the event-provided initialization cells, or read IQM_LIVE_RUNBOOK.md.')
else:
    print('Backend name:', getattr(backend, 'name', type(backend).__name__))


## 3. Compile/dry-run — **no QPU job**

Compare logical gate resources with transpiled native instructions. Qiskit may erase the difference at high optimization levels; start with level 0 and record both.

In [ ]:
if 'backend' in globals():
    preview = compare_on_backend(gauntlet, backend, cfg, shots=256,
                                 probe_qubit=1, optimization_level=0, execute=False)
else:
    print('Preview skipped until backend is initialized; offline unitary check above passed.')


## 4. Explicit physical-hardware opt-in

This action can submit **two actual QPU jobs**, spend tokens/credits and wait in a queue. Only set `ALLOW_HARDWARE_RUN=True` after verifying authorization, shot budget, timeslot and target connectivity. The default is **False**.


In [ ]:
ALLOW_HARDWARE_RUN = False  # Change to True manually ONLY when ready to submit jobs
if ALLOW_HARDWARE_RUN:
    if 'backend' not in globals():
        raise RuntimeError('Initialize your authorized IQM/Qiskit backend first.')
    actual_results = compare_on_backend(gauntlet, backend, cfg, shots=256,
                                        probe_qubit=1, optimization_level=0,
                                        execute=True, save_results=True)
    print('Physical comparison completed; job IDs:', actual_results['job_ids'])
else:
    print('DRY-RUN MODE: no quantum hardware submitted; no experimental QPU data created.')


## 5. Interpret actual measurements carefully

The optional saved `iqm_hardware_results.json` has UTC time, job IDs, shots, counts, native operation counts and depth, and TVD from the ideal distribution. Only treat it as real hardware evidence if step 4 completed with a physical device backend. The project's `full_study/` CSV/JSONL data are **local exact-unitary** experiments. For reproducibility, attach both job IDs and the backend's target details to the final report.